# Pocket-Agent — Judge Demo Notebook

**Runtime: CPU** (Runtime → Change runtime type → CPU)

This notebook downloads the pre-trained model (~3 min) and launches the chatbot demo.

## What this does
1. Installs inference dependencies
2. Clones the repo
3. Downloads `model.gguf` from HuggingFace Hub
4. Smoke-tests `inference.run()`
5. Runs latency benchmark (grader gate check)
6. Launches Gradio demo → **public URL appears at the bottom of Cell 6**

## For custom testing
Use Cell 7 to test any prompt you want directly.

In [ ]:
# ── Cell 1: Install inference dependencies (~2-3 min) ─────────────────────────
%pip install -q \
    'llama-cpp-python>=0.2.57' \
    'gradio>=4.31.0' \
    'transformers>=4.44.0' \
    'peft>=0.12.0' \
    'huggingface_hub>=0.20.0'

print('Inference dependencies installed ✅')

In [ ]:
# ── Cell 2: Clone repo ────────────────────────────────────────────────────────
import os

REPO_URL = 'https://github.com/YOUR_USERNAME/pocket-agent'  # ← same as 01_train

if not os.path.exists('pocket-agent'):
    !git clone {REPO_URL}

%cd pocket-agent
print('Working directory:', os.getcwd())

In [ ]:
# ── Cell 3: Download model from HuggingFace Hub ───────────────────────────────
import os
from huggingface_hub import hf_hub_download

HF_REPO = 'Haseeb9009/Vyrothon'   # ← UPDATE THIS (same as 01_train Cell 8)

os.makedirs('artifacts', exist_ok=True)

if not os.path.exists('artifacts/model.gguf'):
    print('Downloading model.gguf (~220MB)...')
    hf_hub_download(
        repo_id=HF_REPO,
        filename='model.gguf',
        local_dir='artifacts/',
    )
    print('Download complete ✅')
else:
    print('model.gguf already present ✅')

size_mb = os.path.getsize('artifacts/model.gguf') / 1e6
print(f'Model size: {size_mb:.1f} MB')
assert size_mb < 500, f'Model too large: {size_mb:.1f} MB > 500 MB gate'

In [ ]:
# ── Cell 4: Smoke test inference.py ──────────────────────────────────────────
import sys
sys.path.insert(0, '.')
from inference import run

smoke_tests = [
    # (prompt, history, expected_type)
    ('What is the weather in London in Celsius?', [], 'tool_call'),
    ('Convert 100 USD to EUR', [], 'tool_call'),
    ('convert 5 km to miles', [], 'tool_call'),
    ('What do I have tomorrow?', [], 'tool_call'),
    ('tell me a joke', [], 'refusal'),
    ('set a reminder for 8am', [], 'refusal'),
    # Multi-turn
    ('now do GBP instead', [
        {'role': 'user', 'content': 'Convert 100 USD to EUR'},
        {'role': 'assistant', 'content': '<tool_call>{"tool": "currency", "args": {"amount": 100, "from": "USD", "to": "EUR"}}</tool_call>'},
    ], 'tool_call'),
]

print('Smoke test results:')
all_pass = True
for prompt, history, expected_type in smoke_tests:
    result = run(prompt, history)
    is_tool_call = '<tool_call>' in result
    got_type = 'tool_call' if is_tool_call else 'refusal'
    ok = got_type == expected_type
    all_pass = all_pass and ok
    status = '✅' if ok else '❌'
    print(f'  {status} [{got_type:10s}] {prompt[:50]}')
    if not ok:
        print(f'       Expected: {expected_type}  Got: {result[:80]}')

print(f'\nSmoke test: {"PASS ✅" if all_pass else "FAIL ❌"}')

In [ ]:
# ── Cell 5: Latency benchmark (grader gate: mean ≤ 200ms on CPU) ──────────────
import time

benchmark_prompts = [
    'What is the weather in Tokyo in Celsius?',
    'Convert 100 USD to EUR',
    'convert 5 km to miles',
    'What meetings do I have tomorrow?',
    'tell me a joke',
    'weather in Dubai in Fahrenheit',
    '500 PKR to USD',
    'SELECT * FROM users',
    'Schedule a meeting next Monday',
    'what is the capital of France',   # refusal
] * 2  # 20 total (matches grader)

times = []
print('Running 20-prompt latency benchmark...')
for p in benchmark_prompts:
    t = time.time()
    run(p, [])
    times.append((time.time() - t) * 1000)

mean_ms = sum(times) / len(times)
max_ms = max(times)
p95_ms = sorted(times)[18]  # 95th percentile

print(f'Mean   : {mean_ms:.1f}ms  {"✅ PASS" if mean_ms < 200 else "❌ FAIL"}')
print(f'Max    : {max_ms:.1f}ms')
print(f'P95    : {p95_ms:.1f}ms')

if mean_ms >= 200:
    print('WARNING: Latency gate may fail. Check n_threads in inference.py.')

In [ ]:
# ── Cell 6: Launch Gradio demo ────────────────────────────────────────────────
# A public URL will appear below after ~30 seconds
# Judges: click the URL to interact with the chatbot

import subprocess, time, re

proc = subprocess.Popen(
    ['python', 'demo/app.py', '--share'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
    text=True
)

print('Starting Gradio (waiting for public URL)...')
for _ in range(60):
    line = proc.stdout.readline()
    if line:
        print(line.strip())
    if 'gradio.live' in line or 'Running on public URL' in line:
        break
    time.sleep(1)

print('\nThe Gradio demo is running. Keep this cell alive.')
print('To stop: Runtime → Interrupt execution')

In [ ]:
# ── Cell 7: Custom test input (for judges) ────────────────────────────────────
# Edit these values and run this cell to test any prompt

from inference import run
import json, re

# ─── EDIT HERE ───────────────────────────────────────────────────────────────
your_prompt = 'What is the weather in Karachi in Celsius?'

your_history = []  # Leave empty for single-turn.
# For multi-turn, add prior turns like:
# your_history = [
#     {'role': 'user', 'content': 'Convert 100 USD to EUR'},
#     {'role': 'assistant', 'content': '<tool_call>{"tool": "currency", ...}</tool_call>'},
# ]
# ─────────────────────────────────────────────────────────────────────────────

result = run(your_prompt, your_history)
print(f'Prompt : {your_prompt}')
print(f'Result : {result}')

# Parse and pretty-print if it's a tool call
m = re.search(r'<tool_call>(.*?)</tool_call>', result, re.DOTALL)
if m:
    data = json.loads(m.group(1))
    print(f'Tool   : {data["tool"]}')
    print(f'Args   : {json.dumps(data["args"], indent=2)}')
else:
    print('(Plain text refusal — no tool call)')

In [ ]:
# ── Cell 8: Run your own test JSONL file ──────────────────────────────────────
# Upload a .jsonl file (one test per line) and score it.
# Format: {"prompt": "...", "history": [], "expected": "<tool_call>..."}
#
# Expected values:
#   - Tool call: '<tool_call>{"tool": "weather", "args": {"location": "London", "unit": "C"}}</tool_call>'
#   - Refusal:   any plain text string (no <tool_call> tags)

# Option A: Upload via Colab Files panel (left sidebar)
# Option B: Paste path below

TEST_FILE = None  # e.g. '/content/my_tests.jsonl'

if TEST_FILE:
    !python src/eval/score.py --test-file {TEST_FILE} --verbose
else:
    print('Set TEST_FILE to your .jsonl path above, then re-run this cell.')
    print()
    print('Or run the built-in self-test:')
    !python src/eval/score.py